# Project: Blackjack by Monte Carlo control 🃏
Learn to play blackjack from nothing but wins and losses, the classic example from Sutton & Barto ch. 5. Play thousands of hands with an ε-greedy policy, average the final reward for every (state, action) seen, and act greedily on those averages. Then compare your learned strategy with the casino's "basic strategy" heat map.

Topic: [[RL Basics and MDPs]], [[Q-Learning and Policy Gradients]] · guide note: [[Project - Blackjack Monte Carlo Control]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt, collections
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
def draw(rng): return min(int(rng.integers(1, 14)), 10)          # infinite deck: J/Q/K count 10, ace = 1 (usable as 11)
def hand_value(cards):
    s = sum(cards); usable = 1 in cards and s + 10 <= 21
    return (s + 10 if usable else s), usable
def play_hand(policy, rng):
    """Returns the episode as [(state, action, reward)]; state = (player sum, dealer card, usable ace); action 1 = hit, 0 = stick."""
    player, dealer = [draw(rng), draw(rng)], [draw(rng), draw(rng)]; episode = []
    while True:
        total, usable = hand_value(player)
        if total < 12: player.append(draw(rng)); continue              # always hit below 12 (never risky)
        state = (total, dealer[0], usable); a = policy(state)
        if a == 1:
            player.append(draw(rng))
            if hand_value(player)[0] > 21: episode.append((state, a, -1)); return episode
            episode.append((state, a, 0))
        else:
            while hand_value(dealer)[0] < 17: dealer.append(draw(rng))
            p, d = total, hand_value(dealer)[0]
            episode.append((state, a, 1 if d > 21 or p > d else 0 if p == d else -1)); return episode

## 1. ε-greedy action
`Q[state]` is an array `[value of stick, value of hit]`. With probability `eps` act randomly, otherwise take the argmax.

In [ ]:
def eps_greedy(Q, state, eps, rng):
    # TODO 1
    raise NotImplementedError  # TODO 1

## 2. First-visit Monte Carlo update
The return of every step is the episode's **final reward** (no discounting, intermediate rewards are 0). For each (state, action) visited for the *first* time in the episode, increment its count `N` and move `Q` towards the return with the running-average step `1/N`.

In [ ]:
def mc_update(Q, N, episode):
    # TODO 2: incremental average of the final reward for first visits
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_bj():
    Q = collections.defaultdict(lambda: np.zeros(2)); N = collections.defaultdict(lambda: np.zeros(2))
    mc_update(Q, N, [((15, 10, False), 1, 0), ((18, 10, False), 0, 1)]); mc_update(Q, N, [((15, 10, False), 1, -1)])
    return np.allclose(Q[(15, 10, False)], [0, 0.0]) and N[(15, 10, False)][1] == 2 and Q[(18, 10, False)][0] == 1 and eps_greedy(Q, (18, 10, False), 0.0, np.random.default_rng(0)) == 0
check("monte carlo", _t_bj, "Two visits of (15,10,hit) with returns +1 and −1 average to 0; (18,10,stick) got +1.")
def evaluate(policy, n=50000, seed=1):
    rng = np.random.default_rng(seed); return np.mean([play_hand(policy, rng)[-1][2] for _ in range(n)])
if ready("monte carlo"):
    rng = np.random.default_rng(0); Q = collections.defaultdict(lambda: np.zeros(2)); N = collections.defaultdict(lambda: np.zeros(2))
    for i in range(300000): mc_update(Q, N, play_hand(lambda s: eps_greedy(Q, s, 0.1, rng), rng))
    learned = lambda s: int(np.argmax(Q[s]))
    res = {"random": evaluate(lambda s: int(np.random.default_rng(hash(s) % 1000).integers(2))), "mimic the dealer (hit < 17)": evaluate(lambda s: int(s[0] < 17)),
           "learned (MC control)": evaluate(learned)}
    for k, v in res.items(): print(f"{k:<28} expected reward per hand {v:+.3f}")
    grid = np.array([[learned((p, d, False)) for d in range(1, 11)] for p in range(21, 11, -1)])
    plt.imshow(grid, cmap="coolwarm", extent=[0.5, 10.5, 11.5, 21.5]); plt.xlabel("dealer shows (1 = ace)"); plt.ylabel("player sum (no usable ace)")
    plt.title("learned policy: red = hit, blue = stick"); plt.show()
    check("beats simple strategies", lambda: res["learned (MC control)"] > res["mimic the dealer (hit < 17)"] and res["learned (MC control)"] > -0.08,
          "The learned policy should beat 'hit below 17' and lose less than 8 cents per euro.")

Compare the heat map with published *basic strategy*: stick on 12–16 when the dealer shows 2–6 (the dealer is likely to bust), hit otherwise, always stick on 17+. Even the best policy loses a little on average: the house edge.

<details><summary>▶ Solution</summary>

```python
def eps_greedy(Q, state, eps, rng):
    if rng.random() < eps:
        return int(rng.integers(2))
    return int(np.argmax(Q[state]))
```

```python
def mc_update(Q, N, episode):
    G = episode[-1][2]
    seen = set()
    for state, action, _ in episode:
        if (state, action) in seen:
            continue
        seen.add((state, action))
        N[state][action] += 1
        Q[state][action] += (G - Q[state][action]) / N[state][action]
```
</details>

## Stretch goals
- Plot the learned policy *with* a usable ace. Where does it differ?
- Replace Monte Carlo with Q-learning (TD). Which converges faster?
- Add doubling down as a third action.